# Model Training Runner

This notebook drives `src/model_training.py` end-to-end.

**Steps covered:**
1. Resolve project root and configure paths
2. Verify data files exist
3. Run the training script
4. Inspect saved metrics

## Step 1 — Resolve project root and configure paths

`git rev-parse --show-toplevel` gives the absolute repo root regardless of where
Jupyter was launched, so all paths below are stable.

In [2]:
import subprocess
import sys
from pathlib import Path

from src.paths import MODEL_DATA_DIR, MODELS_DIR

PROJECT_ROOT = subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

TRAIN = MODEL_DATA_DIR / "df_train_final.parquet"
VALID = MODEL_DATA_DIR / "df_valid_final.parquet"
TEST = MODEL_DATA_DIR / "df_test_final.parquet"
OUT_DIR = str(MODELS_DIR)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"OUT_DIR      : {OUT_DIR}")

PROJECT_ROOT : D:/AI/Real projects/Academic_Advisor
OUT_DIR      : D:\AI\Real projects\Academic_Advisor\models


## Step 2 — Verify data files exist

In [3]:
for path in [TRAIN, VALID, TEST]:
    p = Path(path)
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {path}")

[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_train_final.parquet
[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_valid_final.parquet
[OK] D:\AI\Real projects\Academic_Advisor\data\model_data\df_test_final.parquet


## Step 3 — Run the training script

Calls `src.model_training.main()` directly so output streams into the notebook.

> The script will:
> - **STEP 8.5** sanity-check `final_mark` (existence, no nulls, values in [0, 100])
> - Train the LightGBM grade regression model (MAE loss)
> - Train the LightGBM pass/fail classifier (binary AUC)
> - Run stratified AUC breakdown (skips segments with only one class)
> - Save `grade_model.lgbm`, `pass_model.lgbm`, and `metrics.json` to `OUT_DIR`

In [4]:
import json

from src.model_training import main

# All arguments default to MODEL_DATA_DIR/df_*_final.parquet and MODELS_DIR
# inside main(), so no sys.argv patching is needed. Pass an explicit empty
# list so Jupyter's own kernel arguments are not parsed; override with e.g.
# main(["--train", str(TRAIN)]) only when deviating from the defaults.
main([])

Loading data …
  train (450465, 71)  valid (156097, 71)  test (110008, 71)

Learning categorical levels (train only) …
  [cat] requirement_type_id: learned 7 levels from train -> [-1, 1, 2, 3, 4, 5, 6]

=== PRE-TRAINING DIAGNOSTICS ===

[1] MODEL_FEATURES count = 39 (expected 39)
     1. prev_gpa_points_clean
     2. start_agpa_points
     3. last_valid_gpa_before_current_semester
     4. start_total_in_credits
     5. start_total_in_courses
     6. total_fail_credits_capped
     7. fail_credit_ratio_capped
     8. is_extreme_fail_history
     9. reg_total_semesters
    10. start_level_ord
    11. start_level_missing
    12. start_semester
    13. prior_interruption_count
    14. consecutive_interruption_count
    15. prev_semester_was_interruption
    16. no_previous_progress
    17. is_first_active_semester
    18. is_first_row_in_timeline
    19. prev_gpa_points_missing
    20. prev_gpa_points_zero
    21. prev_gpa_invalid_zero_case
    22. semester_reg_credits
    23. semester_reg_

## Step 4 — Inspect saved metrics

In [5]:
metrics_path = Path(OUT_DIR) / "metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2))

{
  "m1_pass_classifier": {
    "train": {
      "auc": 0.9055,
      "avg_precision": 0.9764,
      "accuracy": 0.8891,
      "precision": 0.8862,
      "recall": 0.9961,
      "f1": 0.9379,
      "fail_precision": 0.9396,
      "fail_recall": 0.3216,
      "fail_f1": 0.4792,
      "brier": 0.0825,
      "confusion_matrix": {
        "tn": 22992,
        "fp": 48490,
        "fn": 1478,
        "tp": 377505
      }
    },
    "valid": {
      "auc": 0.7807,
      "avg_precision": 0.9677,
      "accuracy": 0.8896,
      "precision": 0.8987,
      "recall": 0.9883,
      "f1": 0.9414,
      "fail_precision": 0.2524,
      "fail_recall": 0.0342,
      "fail_f1": 0.0603,
      "brier": 0.0864,
      "confusion_matrix": {
        "tn": 553,
        "fp": 15594,
        "fn": 1638,
        "tp": 138312
      }
    },
    "test": {
      "auc": 0.7692,
      "avg_precision": 0.97,
      "accuracy": 0.8991,
      "precision": 0.9118,
      "recall": 0.9844,
      "f1": 0.9467,
      "fail_pre

## Alternative — run as CLI command

From the project root in a terminal (all arguments default to
`MODEL_DATA_DIR/df_{train,valid,test}_final.parquet` and `MODELS_DIR`):

```bash
python -m src.model_training
```

Pass `--train/--valid/--test/--out` only to override a default. Set
`ACADEMIC_ADVISOR_DATA_DIR` before running so `src/paths.py` resolves the
intended data root.